In [ ]:
import os
import numpy as np
import pandas as pd

In [1]:
from tool import TOOL
OJ = TOOL()

In [34]:
step = 40
X = []
Y = []

In [ ]:
list_files_csv = os.listdir('DATA_TUC')
for file_csv in list_files_csv:
    with open('DATA_TUC/' + file_csv, 'r') as f:
        data = pd.read_csv(f).to_numpy()
        for i in range(step, data.shape[0]):
            X.append(data[i - step: i, :])
            Y.append(1)

In [ ]:
list_files_csv = os.listdir('DATA_KHONG_TUC')
for file_csv in list_files_csv:
    with open('DATA_KHONG_TUC/' + file_csv, 'r') as f:
        data = pd.read_csv(f).to_numpy()
        for i in range(step, data.shape[0]):
            X.append(data[i - step: i, :])
            Y.append(0)

In [37]:
X = np.array(X)
Y = np.array(Y)
print(X.shape, Y.shape)

(246, 40, 40) (246,)


In [9]:
Y = np.array(Y)
X = np.array(X)

In [10]:
from sklearn.model_selection import train_test_split

xtrain, xtest, ytrain, ytest = train_test_split(X, Y, test_size=0.2)


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [38]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM,Flatten, Dropout, BatchNormalization, Bidirectional, GlobalAveragePooling1D

# Xây dựng mô hình tối ưu hơn
model = Sequential([
    Bidirectional(LSTM(128, activation='tanh', return_sequences=True), input_shape=(40, 40)),
    BatchNormalization(),

    Bidirectional(LSTM(256, activation='tanh', return_sequences=True)),
    BatchNormalization(),

    Bidirectional(LSTM(512, activation='tanh', return_sequences=True)),
    Dropout(0.2),

    Flatten(),

    Dense(16, activation="tanh"),
    Dropout(0.2),
    
    Dense(1, activation="sigmoid")  # Binary classification
])

# Biên dịch mô hình
model.compile(optimizer='adam', loss='categrorical_crossentropy', metrics=['accuracy'])

# In tóm tắt mô hình
model.summary()


c:\Users\levie\.conda\envs\py310\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 40, 256)        │       173,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 40, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 40, 512)        │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 40, 512)        │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 40, 1024)       │     4,198,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 40960)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │       655,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,080,545 (23.20 MB)

 Trainable params: 6,079,009 (23.19 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [ ]:
model.fit(xtrain, ytrain, batch_size= 16, epochs= 10, validation_data= (xtest, ytest))

Epoch 1/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 21s 446ms/step - accuracy: 0.7036 - loss: 0.8496 - val_accuracy: 0.4632 - val_loss: 1.3446
Epoch 2/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 14s 426ms/step - accuracy: 0.8203 - loss: 0.3967 - val_accuracy: 0.4632 - val_loss: 1.8358
Epoch 3/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 14s 407ms/step - accuracy: 0.8287 - loss: 0.4399 - val_accuracy: 0.4632 - val_loss: 2.1617
Epoch 4/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 15s 452ms/step - accuracy: 0.9045 - loss: 0.2301 - val_accuracy: 0.9044 - val_loss: 0.3728
Epoch 5/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 14s 415ms/step - accuracy: 0.8813 - loss: 0.2393 - val_accuracy: 0.4632 - val_loss: 2.0429
Epoch 6/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 16s 466ms/step - accuracy: 0.9037 - loss: 0.2564 - val_accuracy: 0.4632 - val_loss: 2.0602
Epoch 7/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 15s 435ms/step - accuracy: 0.9448 - loss: 0.2198 - val_accuracy: 0.5368 - val_loss: 1.8797
Epoch 8/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 15s 456ms/step - accuracy: 0.9636 - loss: 0.1298 - val_accu

In [ ]:
model.save('model_lstm.h5')